# OAC Catalog ACL Transformer — Silver Layer

**Version:** 2.1 — ADW source and target, admin detection, OCI GenAI enrichments  
**Source:** `arganoadw_oacuser_sh.oacuser.OAC_CATALOG_ACL` (Bronze ADW table)  
**Target:** `arganoadw_oacuser_sh.oacuser.OAC_CATALOG_ACL_SILVER` (Silver ADW table)

---
### Enrichments Applied

**PySpark (v1 — preserved):**
1. `ITEM_PATH_DECODED` — Base64 decode `ITEM_ID` to readable catalog path
2. `CATALOG_TYPE_LABEL` — proper case (`workbooks` → `Workbooks`)
3. `ITEM_MODIFIED_TS` — ISO string → Spark `TimestampType`
4. `PERMISSION_LEVEL` — 6 boolean flags → single readable label
5. `RISK_SCORE` — weighted integer 0–10
6. `RISK_LEVEL` — score bucket label (Critical / High / Medium / Low / None)
7. `ACCOUNT_CATEGORY` — clean principal type label
8. `EXTRACTED_AT_TS` — ISO string → `TimestampType`
9. `HAS_CREATED_DATE` — null quality flag (0/1)
10. `DATA_QUALITY_FLAG` — row-level completeness indicator

**GenAI (v2 — new):**
11. `IS_ADMIN_ACCOUNT` — admin flag (name pattern + Full Control breadth detection)
12. `RISK_NARRATIVE` — plain-English risk summary per account via OCI GenAI (Agent B).
    Admin accounts receive a standard override note — not a risk flag.
13. `INFERRED_OWNER` — suggested owner for orphaned/service-account-owned objects (Agent C)
14. `INFERRED_OWNER_NOTE` — one-sentence rationale from the model

---
### Pre-requisites
1. Bronze notebook completed — `OAC_CATALOG_ACL` exists in ADW
2. `arganoadw_oacuser_sh` External Catalog registered in AIDP
3. `oacuser` schema pre-exists in ADW — do **not** attempt `CREATE SCHEMA`
4. `GENAI_SQL_FUNCTION` set in Section 1 to your registered model function name
5. Spark cluster attached to this notebook

---
### Architecture Note
Reads from and writes to ADW via the AIDP External Catalog — the same proven
pattern as Bronze v1 and Silver v1. No object storage staging layer.
`saveAsTable()` is the only write call; no read-back validation is attempted
after writing as the External Catalog 3-part path does not resolve correctly
in `spark.table()` or `spark.sql()` read-back calls in this environment.
Row and column counts are taken from the in-memory DataFrame instead.

## Section 1 — Imports & Configuration
All configuration defined here — source/target tables, risk weights, admin patterns, and GenAI settings.

**Risk score weight design:**
- `WEIGHT_CHANGE_PERM` (4) — highest risk: can modify other users' access
- `WEIGHT_TAKE_OWN` (3) — can reassign ownership, bypassing access controls
- `WEIGHT_DELETE` (2) — destructive: can permanently remove the item
- `WEIGHT_WRITE` (1) — data modification risk

**Admin detection:**
- `ADMIN_NAME_PATTERNS` — case-insensitive substring match against `ACCOUNT_NAME`. Matching accounts receive `ADMIN_NARRATIVE` and skip GenAI.
- `ADMIN_FC_THRESHOLD` — accounts with Full Control on this many distinct catalog types are flagged as admin regardless of name.

**GenAI function name:**  
Find in AIDP Workbench → Default Catalog → OCI AI Models — the SQL function name is shown next to each registered model. Set `GENAI_SQL_FUNCTION` to match.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, IntegerType, StructType, StructField
import base64
import time
from datetime import datetime, timezone
from collections import defaultdict

# ── Source & Target (ADW External Catalog) ────────────────────
# Both tables live in the same ADW schema. The External Catalog
# connection resolves the 3-part path via AIDP metadata.
# Do NOT attempt CREATE SCHEMA — returns 502 from AIDP Metastore.
AIDP_CATALOG  = 'arganoadw_oacuser_sh'
AIDP_SCHEMA   = 'oacuser'
BRONZE_TABLE  = 'OAC_CATALOG_ACL'
SILVER_TABLE  = 'OAC_CATALOG_ACL_SILVER'

BRONZE_FULL   = f'{AIDP_CATALOG}.{AIDP_SCHEMA}.{BRONZE_TABLE}'
SILVER_FULL   = f'{AIDP_CATALOG}.{AIDP_SCHEMA}.{SILVER_TABLE}'

# ── Risk Score Weights (tune to reflect org risk posture) ─────
WEIGHT_CHANGE_PERM  = 4   # Can change permissions on item
WEIGHT_TAKE_OWN     = 3   # Can take ownership of item
WEIGHT_DELETE       = 2   # Can delete item
WEIGHT_WRITE        = 1   # Can write/edit item

# ── Admin Account Detection ───────────────────────────────────
# Case-insensitive substring match against ACCOUNT_NAME.
# Matching accounts receive ADMIN_NARRATIVE and skip GenAI Agent B.
# Add patterns to match your environment's admin naming conventions.
ADMIN_NAME_PATTERNS = [
    'admin', 'administrator', 'bi_administrator',
    'biserviceadministrator', 'sysadmin', 'oac-admin', 'catalog-admin',
]

# Accounts with Full Control on this many or more distinct catalog
# types are also flagged as admin regardless of name.
ADMIN_FC_THRESHOLD = 3

# Standard narrative applied to all admin accounts — no GenAI call.
ADMIN_NARRATIVE = (
    'This is an administrative or privileged account. '
    'Elevated access across catalog types is expected and by design. '
    'No remediation required. Recommend periodic verification that '
    'this account is actively managed and assigned to the correct owner.'
)

# ── Owner Inference Triggers (Agent C) ───────────────────────
# ITEM_OWNER values matching these patterns (case-insensitive)
# are treated as service/shared accounts and trigger inference.
# A null or empty ITEM_OWNER also triggers inference.
SERVICE_ACCOUNT_PATTERNS = [
    'svc-', 'service-', 'shared-', 'system-', 'oac-svc', 'oacservice',
]

# ── OCI GenAI (AIDP SQL Method) ───────────────────────────────
# GenAI is called via Spark SQL using the model function registered
# in the AIDP Default Catalog. No SDK or Resource Principal needed.
#
# Find your function name:
#   AIDP Workbench → Default Catalog → OCI AI Models
#   The SQL function name is shown next to each registered model.
# ⚠️  ACTION REQUIRED: set to the function name in your instance.
# Registered GenAI function name — confirmed via SHOW FUNCTIONS.
# Called as: ai_generate(prompt_column) — no model name argument needed.
# To switch models, update the Default Catalog AI Model mapping in AIDP Workbench.
GENAI_FUNCTION = 'ai_generate'

print('=' * 55)
print('  SECTION 1 COMPLETE: Imports & Configuration')
print(f'  Source : {BRONZE_FULL}')
print(f'  Target : {SILVER_FULL}')
print(f'  GenAI  : {GENAI_FUNCTION}()')
print(f'  Risk weights: ChangePerm={WEIGHT_CHANGE_PERM}, '
      f'TakeOwn={WEIGHT_TAKE_OWN}, Delete={WEIGHT_DELETE}, Write={WEIGHT_WRITE}')
print('=' * 55)

## Section 2 — Spark Session & Read Bronze
Reads from the Bronze ADW table via the AIDP External Catalog connection. No wallet, no JDBC — Spark resolves the 3-part path through the External Catalog metadata layer.

The Bronze table is the raw, unmodified API extract and the source of truth. It is **never written to from this notebook** — all enrichment goes to the Silver table only.

In [ ]:
spark = SparkSession.builder.appName('oac_acl_silver').getOrCreate()

df_bronze    = spark.table(BRONZE_FULL)
total_bronze = df_bronze.count()

print('=' * 55)
print('  SECTION 2 COMPLETE: Bronze Table Loaded')
print(f'  Source    : {BRONZE_FULL}')
print(f'  Rows read : {total_bronze:,}')
print(f'  Columns   : {len(df_bronze.columns)}')
print('=' * 55)

## Section 3 — UDFs
Four Python functions registered as Spark UDFs for row-level transformation logic.

- **`decode_base64_id`** — reverses the Base64 URL-safe encoding applied during Bronze extraction. e.g. `L3NoYXJlZC9TYWxlcw` → `/shared/Sales`. Returns original value if decode fails.
- **`get_permission_level`** — maps 6 boolean flags to a single label (highest privilege wins): `Full Control` → `Read-Write-Delete` → `Read-Write` → `Read-Only` → `No Access`
- **`get_risk_score`** — weighted integer 0–10 using `WEIGHT_*` constants from Section 1. Re-run this section if weights are changed.
- **`get_account_category`** — maps `ACCOUNT_TYPE` (`ApplicationRole` / `User`) to `Application Role` / `Individual User` for clean OAC report filtering.

In [ ]:
def decode_base64_id(encoded_id):
    if not encoded_id:
        return None
    try:
        padded = encoded_id + '=' * (4 - len(encoded_id) % 4)
        return base64.urlsafe_b64decode(padded).decode('utf-8')
    except Exception:
        return encoded_id   # return original if decode fails


def get_permission_level(read, write, delete, change_perm, take_own):
    if change_perm == 1 or take_own == 1:
        return 'Full Control'
    elif delete == 1:
        return 'Read-Write-Delete'
    elif write == 1:
        return 'Read-Write'
    elif read == 1:
        return 'Read-Only'
    else:
        return 'No Access'


def get_risk_score(write, delete, change_perm, take_own):
    if write is None and delete is None and change_perm is None and take_own is None:
        return 0
    score = 0
    if change_perm == 1: score += WEIGHT_CHANGE_PERM
    if take_own    == 1: score += WEIGHT_TAKE_OWN
    if delete      == 1: score += WEIGHT_DELETE
    if write       == 1: score += WEIGHT_WRITE
    return score


def get_account_category(account_type, account_guid):
    if account_type == 'ApplicationRole':
        return 'Application Role'
    elif account_type == 'User':
        return 'Individual User'
    elif account_guid and '@' in account_guid:
        return 'Individual User'
    else:
        return 'Unknown'


udf_decode_id        = F.udf(decode_base64_id,     StringType())
udf_permission_level = F.udf(get_permission_level,  StringType())
udf_risk_score       = F.udf(get_risk_score,        IntegerType())
udf_account_category = F.udf(get_account_category,  StringType())

print('=' * 55)
print('  SECTION 3 COMPLETE: UDFs Registered')
print('  UDFs: decode_base64_id, get_permission_level,')
print('        get_risk_score, get_account_category')
print('=' * 55)

## Section 4 — PySpark Transformations (10 enrichments)
All 10 enrichments applied in a chained `withColumn()` pattern. Bronze originals first in the final `.select()`, Silver enrichments grouped at the end for clear lineage.

| # | Column | Description |
|---|--------|-------------|
| 1 | `ITEM_PATH_DECODED` | Base64 → readable catalog path |
| 2 | `CATALOG_TYPE_LABEL` | Proper case — `workbooks` → `Workbooks` |
| 3 | `ITEM_MODIFIED_TS` | ISO string → `TimestampType` |
| 4 | `PERMISSION_LEVEL` | 6 flags → readable label |
| 5 | `RISK_SCORE` | Weighted integer 0–10 |
| 6 | `RISK_LEVEL` | Critical / High / Medium / Low / None |
| 7 | `ACCOUNT_CATEGORY` | Application Role / Individual User |
| 8 | `EXTRACTED_AT_TS` | ISO string → `TimestampType` |
| 9 | `HAS_CREATED_DATE` | 0/1 quality flag — `ITEM_CREATED` blank on this OAC instance |
| 10 | `DATA_QUALITY_FLAG` | `MISSING_PRINCIPAL` / `MISSING_PERMISSIONS` / `OK` |

> **`ITEM_CREATED` hooks A–D** are commented out — activate when the field is populated on the target OAC instance.

In [ ]:
df_silver = (
    df_bronze

    # 1. Decode Base64-encoded ITEM_ID to readable catalog path
    .withColumn('ITEM_PATH_DECODED', udf_decode_id(F.col('ITEM_ID')))

    # 2. CATALOG_TYPE_LABEL — proper case for reporting
    .withColumn('CATALOG_TYPE_LABEL', F.initcap(F.col('CATALOG_TYPE')))

    # 3. Cast ITEM_MODIFIED to proper timestamp
    .withColumn('ITEM_MODIFIED_TS', F.to_timestamp(F.col('ITEM_MODIFIED')))

    # 4. PERMISSION_LEVEL — map 6 boolean flags to readable label
    .withColumn('PERMISSION_LEVEL',
        udf_permission_level(
            F.col('PERM_READ'), F.col('PERM_WRITE'), F.col('PERM_DELETE'),
            F.col('PERM_CHANGE_PERM'), F.col('PERM_TAKE_OWN')
        )
    )

    # 5. RISK_SCORE — weighted integer 0–10
    #    Weights defined in Section 1. Re-run Section 3 if weights change.
    .withColumn('RISK_SCORE',
        udf_risk_score(
            F.col('PERM_WRITE'), F.col('PERM_DELETE'),
            F.col('PERM_CHANGE_PERM'), F.col('PERM_TAKE_OWN')
        )
    )

    # 6. RISK_LEVEL — bucket RISK_SCORE into named tiers
    .withColumn('RISK_LEVEL',
        F.when(F.col('RISK_SCORE') >= 8, 'Critical')
         .when(F.col('RISK_SCORE') >= 5, 'High')
         .when(F.col('RISK_SCORE') >= 2, 'Medium')
         .when(F.col('RISK_SCORE') == 0, 'None')
         .otherwise('Low')
    )

    # 7. ACCOUNT_CATEGORY — clean principal type label
    .withColumn('ACCOUNT_CATEGORY',
        udf_account_category(F.col('ACCOUNT_TYPE'), F.col('ACCOUNT_GUID'))
    )

    # 8. Cast EXTRACTED_AT to proper timestamp
    .withColumn('EXTRACTED_AT_TS', F.to_timestamp(F.col('EXTRACTED_AT')))

    # 9. HAS_CREATED_DATE — flag rows missing creation date
    #    ITEM_CREATED is blank on this OAC instance but is a valid API
    #    field on others. The flag is kept active so the column exists
    #    in Silver regardless of instance.
    .withColumn('HAS_CREATED_DATE',
        F.when(
            F.col('ITEM_CREATED').isNull() | (F.col('ITEM_CREATED') == ''),
            F.lit(0)
        ).otherwise(F.lit(1))
    )

    # ── ITEM_CREATED HOOKS (activate when field is populated) ──
    # Hook A: .withColumn('ITEM_CREATED_TS', F.to_timestamp(F.col('ITEM_CREATED')))
    # Hook B: .withColumn('ITEM_AGE_DAYS',   F.datediff(F.col('ITEM_MODIFIED_TS'), F.col('ITEM_CREATED_TS')))
    # Hook C: .withColumn('ITEM_AGE_BUCKET',
    #             F.when(F.col('ITEM_AGE_DAYS') <= 30,  '< 30 days')
    #              .when(F.col('ITEM_AGE_DAYS') <= 90,  '30-90 days')
    #              .when(F.col('ITEM_AGE_DAYS') <= 365, '90-365 days')
    #              .otherwise('> 1 year'))
    # Hook D: add ITEM_CREATED, ITEM_CREATED_TS, ITEM_AGE_DAYS, ITEM_AGE_BUCKET to .select() below
    # ──────────────────────────────────────────────────────────

    # 10. DATA_QUALITY_FLAG — row-level completeness check
    .withColumn('DATA_QUALITY_FLAG',
        F.when(
            F.col('ACCOUNT_GUID').isNull() | F.col('ACCOUNT_NAME').isNull(),
            'MISSING_PRINCIPAL'
        ).when(
            F.col('PERM_READ').isNull() & F.col('PERM_WRITE').isNull(),
            'MISSING_PERMISSIONS'
        ).otherwise('OK')
    )

    # Final column selection — Bronze originals first,
    # enrichments grouped at the end for clear lineage
    .select(
        # Original Bronze columns — preserved unchanged
        F.col('CATALOG_TYPE'),
        F.col('ITEM_ID'),
        F.col('ITEM_NAME'),
        F.col('ITEM_PATH').alias('ITEM_PATH_RAW'),
        F.col('ITEM_OWNER'),
        F.col('ITEM_MODIFIED'),
        F.col('ITEM_CREATED'),
        F.col('ACCOUNT_GUID'),
        F.col('ACCOUNT_TYPE'),
        F.col('ACCOUNT_NAME'),
        F.col('PERM_READ'),
        F.col('PERM_WRITE'),
        F.col('PERM_LIST'),
        F.col('PERM_DELETE'),
        F.col('PERM_CHANGE_PERM'),
        F.col('PERM_TAKE_OWN'),
        F.col('EXTRACTED_AT'),
        # Silver enrichments
        F.col('CATALOG_TYPE_LABEL'),
        F.col('ITEM_PATH_DECODED').alias('ITEM_PATH'),
        F.col('ITEM_MODIFIED_TS'),
        F.col('ACCOUNT_CATEGORY'),
        F.col('PERMISSION_LEVEL'),
        F.col('RISK_SCORE'),
        F.col('RISK_LEVEL'),
        F.col('HAS_CREATED_DATE'),
        F.col('DATA_QUALITY_FLAG'),
        F.col('EXTRACTED_AT_TS')
    )
)

total_silver = df_silver.count()
ok_rows      = df_silver.filter(F.col('DATA_QUALITY_FLAG') == 'OK').count()

print('=' * 55)
print('  SECTION 4 COMPLETE: Transformations Applied')
print(f'  Total rows   : {total_silver:,}')
print(f'  Quality OK   : {ok_rows:,}')
print(f'  Flagged rows : {total_silver - ok_rows:,}')
print('=' * 55)

## Section 5 — Quality Summary
Three distribution tables printed **before writing** — validate enrichment results look correct before committing data.

If distributions look wrong (e.g. all rows showing `No Access`, unexpected `RISK_LEVEL` values), stop here, investigate the Bronze data, and re-run Section 4.

**Expected distributions based on current data:**
- `PERMISSION_LEVEL`: majority `Read-Only` or `Full Control`
- `RISK_LEVEL`: mix of `High` and `None`
- `ACCOUNT_CATEGORY`: `Application Role` dominant

> `df.show()` is retained here intentionally — this runs on aggregated distributions (small row count), not the full Silver DataFrame. Safe at any scale.

In [ ]:
print('\n📊 Permission Level Distribution:')
df_silver.groupBy('PERMISSION_LEVEL').count().orderBy(F.col('count').desc()).show()

print('📊 Risk Level Distribution:')
df_silver.groupBy('RISK_LEVEL').count().orderBy(F.col('count').desc()).show()

print('📊 Account Category Distribution:')
df_silver.groupBy('ACCOUNT_CATEGORY').count().orderBy(F.col('count').desc()).show()

print('=' * 55)
print('  SECTION 5 COMPLETE: Quality Summary Printed')
print('  Review distributions above before proceeding.')
print('=' * 55)

## Section 6 — Admin Account Detection
Flags admin accounts **before** GenAI enrichment so they receive a standard note rather than a false-positive risk alert.

**Why is this step necessary?**  
Admin accounts legitimately hold Full Control across catalog types as part of their role. Without flagging, Agent B would generate high-risk alerts for expected behaviour — reducing trust in the Risk Dashboard.

**Two detection mechanisms (either fires the flag):**
1. **Name-based** — case-insensitive substring match against `ACCOUNT_NAME` using `ADMIN_NAME_PATTERNS`
2. **Breadth-based** — Full Control on `ADMIN_FC_THRESHOLD`+ distinct catalog types, regardless of name

**Output column:** `IS_ADMIN_ACCOUNT` (boolean)
- `True` → receives `ADMIN_NARRATIVE` constant, skips GenAI in Section 8
- `False` → processed by Agent B

In [ ]:
# Name-based condition — OR across all configured patterns
admin_name_condition = F.lit(False)
for pattern in ADMIN_NAME_PATTERNS:
    admin_name_condition = admin_name_condition | F.lower(F.col('ACCOUNT_NAME')).contains(pattern.lower())

# Breadth-based — Full Control on ADMIN_FC_THRESHOLD+ distinct catalog types
admin_breadth_df = (
    df_silver
    .filter(F.col('PERMISSION_LEVEL') == 'Full Control')
    .groupBy('ACCOUNT_NAME')
    .agg(F.countDistinct('CATALOG_TYPE').alias('FC_TYPE_COUNT'))
    .filter(F.col('FC_TYPE_COUNT') >= ADMIN_FC_THRESHOLD)
    .select('ACCOUNT_NAME', F.lit(True).alias('IS_BREADTH_ADMIN'))
)

# Join breadth flag, combine with name flag, drop interim column
df_silver = df_silver.join(admin_breadth_df, on='ACCOUNT_NAME', how='left')
df_silver = df_silver.withColumn(
    'IS_ADMIN_ACCOUNT',
    admin_name_condition | (F.col('IS_BREADTH_ADMIN') == True)
).drop('IS_BREADTH_ADMIN')

admin_count = df_silver.filter(F.col('IS_ADMIN_ACCOUNT') == True).select('ACCOUNT_NAME').distinct().count()
total_count = df_silver.select('ACCOUNT_NAME').distinct().count()

print('=' * 55)
print('  SECTION 6 COMPLETE: Admin Account Detection')
print(f'  Total distinct accounts  : {total_count:,}')
print(f'  Admin accounts flagged   : {admin_count:,}')
print(f'  Non-admin (GenAI input)  : {total_count - admin_count:,}')
print('=' * 55)

## Section 7 — OCI GenAI Setup (`ai_generate`)
`ai_generate` is the registered function in this AIDP instance — confirmed via `SHOW FUNCTIONS`. It takes a single prompt column argument and routes the call to OCI GenAI internally using AIDP's service account.

```python
# PySpark — the pattern used in this notebook
df.withColumn('GEN_RESPONSE', F.expr("ai_generate(GEN_PROMPT)"))

# SQL equivalent
SELECT ai_generate(GEN_PROMPT) AS GEN_RESPONSE FROM prompt_view
```

No model name argument, no SDK, no Resource Principal required.

**`call_genai()`** accepts a `{key: prompt}` dict, builds a typed DataFrame with `GEN_KEY` and `GEN_PROMPT` columns, calls `ai_generate()` via `F.expr()`, and returns a `{key: response}` dict. Returns `[GENAI_ERROR]` on failure so downstream joins complete without raising.

In [ ]:
def call_genai(prompts_dict):
    """
    Call OCI GenAI via AIDP ai_generate() for a dict of {key: prompt}.
    Returns a dict of {key: generated_text}.

    ai_generate() is the registered function in this AIDP instance,
    confirmed via SHOW FUNCTIONS. It takes a single prompt column —
    no model name argument. AIDP routes the call internally.
    """
    if not prompts_dict:
        return {}

    schema = StructType([
        StructField('GEN_KEY',    StringType(), False),
        StructField('GEN_PROMPT', StringType(), False),
    ])
    df_prompts = spark.createDataFrame(
        [(key, prompt) for key, prompt in prompts_dict.items()],
        schema=schema
    )

    try:
        df_result = df_prompts.withColumn(
            'GEN_RESPONSE',
            F.expr(f'{GENAI_FUNCTION}(GEN_PROMPT)')
        )
        return {
            row.GEN_KEY: (row.GEN_RESPONSE or '').strip()
            for row in df_result.collect()
        }
    except Exception as e:
        err = f'[GENAI_ERROR] {str(e)[:200]}'
        print(f'  ⚠️  {GENAI_FUNCTION}() call failed: {err}')
        return {key: err for key in prompts_dict}


print('=' * 55)
print('  SECTION 7 COMPLETE: ai_generate Ready')
print(f'  Function : {GENAI_FUNCTION}()')
print('  Auth     : AIDP internal (no Resource Principal needed)')
print('=' * 55)

## Section 8 — Agent B: Risk Narrative per Account
Generates a plain-English 2–3 sentence risk summary for each distinct `ACCOUNT_NAME` using OCI GenAI via `spark.sql()`.

**Admin accounts** (flagged in Section 6) are excluded — they receive `ADMIN_NARRATIVE` directly with no GenAI call.

**Batching strategy:** All non-admin account prompts are built into a single DataFrame and passed to `call_genai_sql()` in one Spark SQL operation — more efficient than individual SDK calls.

**Output column:** `RISK_NARRATIVE`
- Admin accounts → `ADMIN_NARRATIVE` constant
- Non-admin accounts → GenAI-generated 2–3 sentence summary
- No profile data → `"No profile available"`
- Failed calls → `[GENAI_ERROR]` prefixed message

The narrative is joined back on `ACCOUNT_NAME` — every row for a given account receives the same narrative.

In [ ]:
def build_risk_narrative_prompt(account_name, account_category, profile):
    profile_lines = ' | '.join([
        f"{p['catalog_type_label']}: {p['permission_level']} ({p['item_count']} object(s), risk: {p['risk_level']})"
        for p in profile
    ])
    return (
        f'You are an Oracle Analytics Cloud security analyst. '
        f'Write a 2-3 sentence plain-English risk assessment for the '
        f"{account_category.lower()} account '{account_name}'. "
        f'Be specific and factual based only on this permission profile. '
        f'Do not use bullet points. '
        f'Permission profile: {profile_lines}. '
        f'Risk assessment:'
    )


print('\n[AGENT B] Building risk narratives via AIDP GenAI SQL...')

# ── Separate admin and non-admin accounts ─────────────────────
admin_accounts = set(
    row.ACCOUNT_NAME
    for row in df_silver.filter(F.col('IS_ADMIN_ACCOUNT') == True)
                        .select('ACCOUNT_NAME').distinct().collect()
    if row.ACCOUNT_NAME
)

# ── Build permission profiles for non-admin accounts ──────────
profile_rows = (
    df_silver
    .filter(F.col('IS_ADMIN_ACCOUNT') == False)
    .filter(F.col('ACCOUNT_NAME').isNotNull())
    .groupBy('ACCOUNT_NAME', 'ACCOUNT_CATEGORY', 'CATALOG_TYPE_LABEL', 'PERMISSION_LEVEL', 'RISK_LEVEL')
    .agg(F.count('*').alias('item_count'))
    .collect()
)

account_profiles = defaultdict(lambda: {'category': 'Unknown', 'permissions': []})
for row in profile_rows:
    account_profiles[row.ACCOUNT_NAME]['category'] = row.ACCOUNT_CATEGORY
    account_profiles[row.ACCOUNT_NAME]['permissions'].append({
        'catalog_type_label': row.CATALOG_TYPE_LABEL,
        'permission_level':   row.PERMISSION_LEVEL,
        'risk_level':         row.RISK_LEVEL,
        'item_count':         row.item_count,
    })

# ── Build prompt dict — one entry per non-admin account ───────
prompt_dict = {
    acct: build_risk_narrative_prompt(acct, prof['category'], prof['permissions'])
    for acct, prof in account_profiles.items()
}

print(f'  Admin accounts  : {len(admin_accounts)} — standard narrative applied')
print(f'  Non-admin       : {len(prompt_dict)} — calling OCI GenAI via SQL...')

# ── Call GenAI for all non-admin prompts in one SQL operation ──
genai_results = call_genai(prompt_dict)

# ── Assemble final narrative lookup ──────────────────────────
narrative_lookup = {acct: ADMIN_NARRATIVE for acct in admin_accounts}
narrative_lookup.update(genai_results)

# ── Join narratives back to df_silver on ACCOUNT_NAME ─────────
narrative_schema = StructType([
    StructField('ACCOUNT_NAME',  StringType(), True),
    StructField('RISK_NARRATIVE', StringType(), True),
])
narrative_df = spark.createDataFrame(
    [(name, text) for name, text in narrative_lookup.items()],
    schema=narrative_schema
)

df_silver = df_silver.join(narrative_df, on='ACCOUNT_NAME', how='left')
df_silver = df_silver.fillna({'RISK_NARRATIVE': 'No profile available'})

print('=' * 55)
print('  SECTION 8 COMPLETE: Agent B — Risk Narratives Done')
print(f'  Admin (override) : {len(admin_accounts)}')
print(f'  GenAI generated  : {len(genai_results)}')
print('=' * 55)

## Section 9 — Agent C: Owner Inference for Orphaned Objects
For catalog objects where `ITEM_OWNER` is null, empty, or matches a service/shared account pattern, calls OCI GenAI via `spark.sql()` to suggest the most likely responsible owner.

**Trigger conditions:**
1. `ITEM_OWNER` is null or empty
2. `ITEM_OWNER` matches any `SERVICE_ACCOUNT_PATTERNS`

**Prompt context:** object name, catalog type, decoded path, and up to 10 account names with access — the account list often hints at likely ownership.

**Zero-orphan guard:** If no orphaned objects are found, `INFERRED_OWNER` and `INFERRED_OWNER_NOTE` are added as literal columns with default values — `spark.createDataFrame()` is not called on an empty list.

**Output columns:**
- `INFERRED_OWNER` — `"See note"`, `"Unable to infer"`, `"Error"`, or `"N/A — Owner on record"`
- `INFERRED_OWNER_NOTE` — one-sentence rationale from the model

In [ ]:
def build_owner_inference_prompt(item_name, item_path, catalog_type_label, subjects):
    subjects_str = ', '.join(subjects[:10]) if subjects else 'none on record'
    return (
        f'You are an Oracle Analytics Cloud governance analyst. '
        f'The following catalog object has no identifiable human owner. '
        f'Suggest the most likely responsible owner in one sentence and explain your reasoning. '
        f"If you cannot infer an owner, say 'Unable to infer' and briefly explain why. "
        f'Object name: {item_name}. '
        f'Object type: {catalog_type_label}. '
        f'Object path: {item_path}. '
        f'Accounts with access: {subjects_str}. '
        f'Suggested owner and rationale:'
    )


print('\n[AGENT C] Identifying orphaned objects for owner inference...')

# ── Find orphaned / service-account-owned objects ─────────────
orphan_df = (
    df_silver
    .filter(
        F.col('ITEM_OWNER').isNull()
        | (F.col('ITEM_OWNER') == '')
        | F.col('ITEM_OWNER').rlike('(?i)(' + '|'.join(SERVICE_ACCOUNT_PATTERNS) + ')')
    )
    .select('ITEM_ID', 'ITEM_NAME', 'ITEM_PATH', 'CATALOG_TYPE_LABEL', 'ACCOUNT_NAME', 'ITEM_OWNER')
    .distinct()
)

orphan_count = orphan_df.select('ITEM_ID').distinct().count()
print(f'  Orphaned/service-owned objects: {orphan_count}')

if orphan_count == 0:
    # No orphaned objects — add columns directly as literals.
    # Skips createDataFrame() to avoid empty-schema inference error.
    df_silver = (df_silver
        .withColumn('INFERRED_OWNER',      F.lit('N/A — Owner on record'))
        .withColumn('INFERRED_OWNER_NOTE', F.lit(''))
    )
    print('  No orphaned objects — columns added with default values.')

else:
    # ── Build per-object subject lists and metadata ───────────
    orphan_rows     = orphan_df.collect()
    object_subjects = defaultdict(list)
    object_meta     = {}

    for row in orphan_rows:
        iid = row.ITEM_ID
        if row.ACCOUNT_NAME:
            object_subjects[iid].append(row.ACCOUNT_NAME)
        object_meta[iid] = {
            'item_name':          row.ITEM_NAME          or '',
            'item_path':          row.ITEM_PATH           or '',
            'catalog_type_label': row.CATALOG_TYPE_LABEL  or '',
        }

    # ── Build prompt dict keyed on ITEM_ID ────────────────────
    owner_prompt_dict = {
        item_id: build_owner_inference_prompt(
            meta['item_name'], meta['item_path'], meta['catalog_type_label'],
            object_subjects.get(item_id, [])
        )
        for item_id, meta in object_meta.items()
    }

    print(f'  Calling OCI GenAI for {len(owner_prompt_dict)} object(s)...')
    owner_results = call_genai(owner_prompt_dict)

    # ── Parse responses ───────────────────────────────────────
    inference_lookup = {}
    for item_id, response in owner_results.items():
        if response.lower().startswith('unable to infer'):
            inference_lookup[item_id] = ('Unable to infer', response)
        elif response.startswith('[GENAI_ERROR]'):
            inference_lookup[item_id] = ('Error', response)
        else:
            inference_lookup[item_id] = ('See note', response)

    # ── Join back to df_silver on ITEM_ID ─────────────────────
    inference_schema = StructType([
        StructField('ITEM_ID',             StringType(), True),
        StructField('INFERRED_OWNER',      StringType(), True),
        StructField('INFERRED_OWNER_NOTE', StringType(), True),
    ])
    inference_df = spark.createDataFrame(
        [(iid, owner, note) for iid, (owner, note) in inference_lookup.items()],
        schema=inference_schema
    )

    df_silver = df_silver.join(inference_df, on='ITEM_ID', how='left')
    df_silver = df_silver.fillna({
        'INFERRED_OWNER':      'N/A — Owner on record',
        'INFERRED_OWNER_NOTE': ''
    })
    print(f'  Inferences generated: {len(inference_lookup)}')

print('=' * 55)
print('  SECTION 9 COMPLETE: Agent C — Owner Inference Done')
print(f'  Total Silver columns: {len(df_silver.columns)}')
print('=' * 55)

## Section 10 — Write Silver Table
Writes the enriched Silver DataFrame to ADW via the AIDP External Catalog connection. Same proven pattern as Bronze v1 and Silver v1 — no wallet, no JDBC, no DDL.

**Write strategy:** Full overwrite on every run. Silver is always fully regenerated from Bronze — there is no partial state to preserve.

**`overwriteSchema=true`** — allows column additions between runs without `DROP TABLE`. Safe here because Silver is completely rebuilt each time.

**Validation:** Row and column counts are taken from the in-memory `df_silver` DataFrame. No read-back from ADW is attempted — the External Catalog 3-part path does not resolve correctly in post-write `spark.table()` or `spark.sql()` calls in this AIDP environment. If `saveAsTable()` does not raise, the write succeeded.

In [ ]:
(df_silver.write
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(SILVER_FULL))

# Validation: counts from in-memory DataFrame.
# No read-back attempted — External Catalog 3-part path does not
# resolve correctly post-write in spark.table() / spark.sql() here.
print('=' * 55)
print('  SECTION 10 COMPLETE: Silver Table Written')
print(f'  Table   : {SILVER_FULL}')
print(f'  Rows    : {df_silver.count():,}')
print(f'  Columns : {len(df_silver.columns)}')
print(f'  Status  : Queryable in AIDP Master Catalog + OAC')
print('=' * 55)

print('\n' + '=' * 60)
print('  🏁 SILVER PIPELINE COMPLETE')
print('=' * 60)